<a href="https://colab.research.google.com/github/hiiamlars/master_thesis/blob/main/llm_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [15]:
# @title Dependencies

# Installation
! pip install pandas pyarrow -q
! pip install numpy -q
! pip install openai -q
! pip install groq -q

# First-party installations
import itertools
import re
import json
import random
import time
from typing import Dict, Any, Tuple, Optional, Literal, List
from dataclasses import dataclass
import os

# Third-party installations
from google.colab import drive
import pandas as pd
import numpy as np
from openai import OpenAI
from openai import APIError
from groq import Groq

print("\nInstalling and loading dependencies complete!")


Installing and loading dependencies complete!


In [16]:
# @title Paths and definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Thesis/"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ensured working directory exists at: /content/drive/MyDrive/Thesis/.
Ensured dataset directory exists at: /content/drive/MyDrive/Thesis/iclr/final.
Ensured raw directory exists at: /content/drive/MyDrive/Thesis/llm_reviews.


In [17]:
# Models using the OpenAI-API (as used by Cummins)
OPENAI_MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-08-06",
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125"
]

# Models using the Groq-API (as used by Cummins)
GROQ_MODELS = [
    "llama-3.3-70b-versatile",
    "deepseek-r1-distill-llama-70b"
]

# Models who can vary by reasoning effort
REASONING_MODELS = {
    "o4-mini-2025-04-16",
    "gpt-5-mini-2025-08-07",
    "o3-mini-2025-01-31",
    "gpt-5-nano-2025-08-07"
}

# Define temperature parameters (as used by Cummins)
TEMPS = [0.0, 0.5, 1.0, 1.5]

# Define reasoning parameters (as used by Cummins)
REASONING = ["low", "high"]

# Define prompt style (CoT) (as used by Li et al. 2025)
CoT = [
    "",
    "explain your thought process step by step"
]

# Define scope
SCOPE = [
  "abstract",
  "full_text"
]

# Define the number of reviews to generate per paper for each combo
NUM_REVIEWS_PER_PAPER = 2

# Create data class for parameter combinations
@dataclass
class ParamCombo:
    model: str
    temperature: Optional[float]
    reasoning_effort: Optional[Literal[*REASONING]]
    chain_of_thought: Literal[*CoT]
    scope: Literal["abstract", "full_text"]

# Define the full result path
RESULTS_PATH = os.path.join(RAW_DIR, "llm_sim_results.csv")

# Define the full log path
LOG_PATH = os.path.join(RAW_DIR, "llm_sim_progress.csv")

SIMULATION_ACTIVE = True # SIMULATION_ACTIVE = False activates the API calls

print("\nDefintions complete!")


Defintions complete!


In [18]:
# @title APIs

# User prompt
PROMPT_TEMPLATE = """

Here is the text of a paper:

{{text}}

Create an ICLR-style review following this specific structure:

# Summary Of The Paper

Summarize the paper’s main contributions, methodology, and findings.

# Strength And Weaknesses

Analyze the paper’s contributions based on your notes.

# Clarity, Quality, Novelty And Reproducibility

Evaluate based on your notes.

# Summary Of The Review

Provide a 2-3 sentence distillation of your overall assessment.

# Correctness

Rate on a scale of 1-5.

# Technical Novelty And Significance

Rate on a scale of 1-5.

# Empirical Novelty And Significance

Rate on a scale of 1-5.

Maintain a professional tone throughout. Base your review entirely on your reading notes.

""" # Adopted and adjusted from Robertson and Koyejo (2025); how should I implement the first prompt about note taking?

# OpenAI API-key
OPENAI_KEY = "api_key_placeholder"

# Groq API-key
GROQ_KEY = "api_key_placeholder"

# Groq URL
GROQ_URL ="https://api.groq.com/openai/v1"

In [19]:
# @title Load data

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(DATASET_DIR, "dataset_paper.parquet"))

# Check dataset_paper
print(dataset_paper.head(3))

# Select 'paper_id', 'abstract' and the target column 'parsed_text'
paper_content = dataset_paper[['paper_id', 'abstract', 'parsed_text']]

# Prepare Dict for later applications
targets = paper_content.to_dict('records')

      paper_id                                            pdf_url  \
0  vuD2xEtxZcj  https://openreview.net/pdf/b7b54047fdaf97f5057...   
1  WoByU5W5te0  https://openreview.net/pdf/e0646a302829ac0b05b...   
2  XZRmNjUMj0c  https://openreview.net/pdf/20516df845666b5c362...   

                                            abstract  \
0  In deep learning, fine-grained N:M sparsity re...   
1  We present a novel method to regularizes neura...   
2  Neural Architecture Search (NAS) is a fast-dev...   

                                         parsed_text  \
0  INTRODUCTION Pruning Deep Neural Networks (DNN...   
1  INTRODUCTION Recently, representing a 3D scene...   
2  INTRODUCTION Neural Architecture Search (NAS) ...   

                                       human_review1  \
0  Summary Of The Paper:\n\nThe paper aims to acc...   
1  Summary Of The Paper:\n\nThis paper proposes a...   
2  Summary Of The Paper:\n\nThis paper proposes a...   

                                       human_rev

# API

In [20]:
# @title Response scheme

def _schema_to_tool(chain_of_thought: str) -> Dict[str, Any]:

    return {
        "type": "function",
        "function": {
            "name": "response_format",
            "description": "The function used to return the single, structured text response.",
            "parameters": {
                "type": "object",
                "properties": {
                    "final_response": {
                        "type": "string",
                        "description": (
                            f"Generate the complete, continuous peer-review based on the provided content. {chain_of_thought}"
                        )
                    }
                },
                "required": ["final_response"]
            }
        }
    }

In [21]:
# @title Response handling

# Function to parse JSON from a string and strip DeepSeek R1 reasoning if present
def _extract_json(text: str, model_name: str = "") -> dict:
    """Extract and parse a JSON object from a potentially messy API response"""
    if not text:
        return {}

    # DeepSeek R1: strip 'thinking' section if present
    if "deepseek-r1-distill-llama-70b" in model_name:
        # Common pattern: <think>...</think>{...} or reasoning text before JSON
        if "</think>" in text:
            text = text.split("</think>", 1)[-1]
        elif "<think>" in text:
            text = text.split("<think>", 1)[-1]
        # Otherwise, try to cut at the first { or [ after any long text
        if "{" in text or "[" in text:
            idx = min(
                [i for i in [text.find("{"), text.find("[")] if i != -1]
            )
            text = text[idx:]

    # Strip code fences if present
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    # Find first {...}
    m = re.search(r"(\{.*\})", text, flags=re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    try:
        return json.loads(text)
    except Exception:
        return {}

In [22]:
# @title Define LLMClient

# Fake clients for testing
class LLMClient:
    def __init__(self, simulate: bool = True, seed: int = 7):
        """Initializes LLMClient with defaults"""
        self.simulate = simulate
        self.rng = random.Random(seed)

        # Placeholders for real clients when simulate=False:
        self._openai = None
        self._groq = None

    def _maybe_init_clients(self):
        """Starts simulation or API clients"""
        if self.simulate:
            return
        self._openai = OpenAI(api_key=OPENAI_KEY)
        self._groq = Groq(api_key=GROQ_KEY)

    def call(
        self,
        model: str,
        prompt: str,
        temperature: Optional[float],
        reasoning_effort: Optional[str],
        chain_of_thought: Optional[str],
        max_retries: int = 3,
        retry_delay: float = 1.5,
        ) -> Tuple[str, Dict]:

        """Defines the API calls / simulations according to the input parameters and API configuration"""

        self._maybe_init_clients()

        # Random values for test sims
        if self.simulate:

          # Construct artificial output
          raw = {
              "model": model,
              "temperature": temperature,
              "reasoning_effort": reasoning_effort,
              "chain_of_thought": chain_of_thought,
              "simulated_review": (
                  f"This is a simulated review: Model='{model}', Temp={temperature}, "
                  f"Effort='{reasoning_effort}', chain_of_thought='{chain_of_thought}'."
              )
          }

          # Construct output parameter
          structured = json.dumps(raw, indent=2)

          # Immediately return the raw string and the structured dictionary
          return structured, raw

        provider = "groq" if model in GROQ_MODELS else "openai"

        for attempt in range(1, max_retries + 1):
            try:
                if provider == "openai":
                    # If an OpenAI "reasoning" model, use Responses API
                    # And rely on the contract + normalizer.
                    # Otherwise, use Chat Completions with tool-calling to enforce structure strictly.
                    if model in REASONING_MODELS:
                        kwargs = {}
                        if reasoning_effort:
                            kwargs["reasoning"] = {"effort": reasoning_effort}
                        if temperature is not None:
                            kwargs["temperature"] = float(temperature)

                        # Instruction since we can't pass response_format:
                        contract = "Return exactly ONE JSON object" # Helper system prompt
                        resp = self._openai.responses.create(
                            model=model,
                            input=(prompt + contract),
                            **kwargs,
                        )
                        raw = getattr(resp, "output_text", None) or str(resp)

                    else:
                        # Chat Completions + tool calling (strict) for non-reasoning OpenAI models
                        tools = [_schema_to_tool(chain_of_thought)]

                        ckwargs = {}
                        if temperature is not None:
                            ckwargs["temperature"] = float(temperature)

                        # Add nudge to return only a tool call
                        system_msg = {
                            "role": "system", # System prompt
                            "content": "You are a parser. Call the function with exactly one JSON object that includes a text."
                        }

                        msgs = [system_msg, {"role": "user", "content": prompt}]
                        resp = self._openai.chat.completions.create(
                            model=model,
                            messages=msgs,
                            tools=tools if tools else None,
                            **ckwargs,
                        )

                        choice = resp.choices[0]
                        # Prefer tool call arguments (strict JSON)
                        if choice.message.tool_calls:
                            args = choice.message.tool_calls[0].function.arguments
                            raw = args  # Raw JSON text
                        else:
                            # Fallback to message content (should be JSON anyway)
                            raw = choice.message.content or ""

                else:
                    contract = "Return exactly ONE JSON object" # Helper system prompt
                    prompt2 = prompt + contract
                    gkwargs = {}
                    if temperature is not None and model not in REASONING_MODELS:
                        gkwargs["temperature"] = float(temperature)
                    resp = self._groq.chat.completions.create(
                        model=model,
                        messages=[{"role": "user", "content": prompt2}],
                        **gkwargs,
                    )
                    raw = resp.choices[0].message.content.strip()

                # Parse + normalize
                parsed_raw = _extract_json(raw, model_name=model)
                return raw, parsed_raw # Further downstream this might need the 'normalized_raw' key

            except Exception as e:
                print(f"[LLM ERROR] provider={provider} model={model} attempt={attempt} -> {e}", flush=True)
                if attempt == max_retries:
                    err_stub = f"ERROR: {type(e).__name__}: {e}"
                    return err_stub, {}

            time.sleep(retry_delay)

# Logging handlers

In [23]:
def _fmt_combo(combo: ParamCombo) -> str:
    parts = []
    if combo.model is not None:
        parts.append(f"model={combo.model}")
    if combo.temperature is not None:
        parts.append(f"temperature={combo.temperature}")
    if combo.reasoning_effort is not None: # Note: reasoning_effort can be 'low' or 'high', not None
        parts.append(f"reasoning_effort={combo.reasoning_effort}")
    # Chain_of_thought can be an empty string, which should be included
    parts.append(f"chain_of_thought={combo.chain_of_thought}")
    parts.append(f"scope={combo.scope}")
    return ", ".join(parts)

def log_call(doc_id: str, combo: ParamCombo, **context: Dict[str, Any]):
    ctx = ", ".join(f"{k}={v}" for k, v in context.items() if v is not None)
    msg = f"[LLM CALL] paper={doc_id} | {_fmt_combo(combo)}"
    if ctx:
        msg += f" | {ctx}"
    print(msg, flush=True)

# Define simulation

In [24]:
# Define main simulation workflow
def simulate_parameter_combo(
    combo: ParamCombo,
    review_targets: List[Dict],
    client: Optional[LLMClient] = None,
) -> pd.DataFrame:

    """
    Iterates through the papers, creates the final user prompt, takes the configurational setting,
    calls the API/simulation and collects the resulting reviews in a pandas DataFrame
    """

    # If client specified then use client, otherwise fake client for test
    client = client or LLMClient(simulate=True)

    records = []
    for review_item in review_targets:
      # Select content based on scope
      if combo.scope == "full_text":
          paper_content_for_llm = review_item['parsed_text']
      elif combo.scope == "abstract":
          paper_content_for_llm = review_item['abstract']
      else:
          raise ValueError(f"Invalid scope: {combo.scope}. Must be 'full_text' or 'abstract'.")

      # Construct the complete prompt by formatting the PROMPT_TEMPLATE
      # The {{text}} placeholder in PROMPT_TEMPLATE is replaced with the actual paper content
      complete_prompt = PROMPT_TEMPLATE.replace("{{text}}", paper_content_for_llm)

      # Append chain_of_thought instruction to the complete prompt
      if combo.chain_of_thought:
          complete_prompt += f"\n\n{combo.chain_of_thought}"

      doc_id = review_item["paper_id"]

      # Initialize a dictionary for the current paper's results, including combo parameters
      current_paper_record = {
          "paper_id": doc_id,
          "model": combo.model,
          "temperature": combo.temperature,
          "reasoning_effort": combo.reasoning_effort,
          "chain_of_thought": combo.chain_of_thought,
          "scope": combo.scope
      }

      # Loop to generate multiple reviews for each paper and parameter combo
      for review_idx in range(NUM_REVIEWS_PER_PAPER): # Use global constant
          # Log with 1-based index for clarity in logs, though internal loop is 0-based
          log_call(doc_id, combo, review_number=review_idx + 1)

          raw, parsed = client.call(
              model=combo.model,
              prompt=complete_prompt,
              temperature=(combo.temperature if combo.model not in REASONING_MODELS else None),
              reasoning_effort=(combo.reasoning_effort if combo.model in REASONING_MODELS else None),
              chain_of_thought=combo.chain_of_thought
          )

          # Store each review in a new column, e.g., 'llm_review_1', 'llm_review_2'
          current_paper_record[f"llm_review_{review_idx + 1}"] = raw # Changed prefix here

      # Append the single record for the current paper to the list of records
      records.append(current_paper_record)

    df = pd.DataFrame.from_records(records)
    return df

# Prepare configurations

In [25]:
# Complete list of models
MODELS = OPENAI_MODELS + GROQ_MODELS

# Generates every single combination
raw_grid = list(itertools.product(MODELS, TEMPS, REASONING, CoT, SCOPE))
grid_df = pd.DataFrame(raw_grid, columns=["model", "temperature", "reasoning_effort", "chain_of_thought", "scope"])

# Conditional deletion whether a model has a temperature or reasoning parameter
grid_df["reasoning_effort"] = np.where(
    ~grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["reasoning_effort"]
)
grid_df["temperature"] = np.where(
    grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["temperature"]
)

# Drop duplicates
experiment_config = grid_df.drop_duplicates(ignore_index=True)

print(f"Total valid parameter combos: {len(experiment_config)}")
display(experiment_config)

print("\nConfiguration table completed!")

Total valid parameter combos: 112


,model,temperature,reasoning_effort,chain_of_thought,scope
0,gpt-5-mini-2025-08-07,NaN,low,,abstract
1,gpt-5-mini-2025-08-07,NaN,low,,full_text
2,gpt-5-mini-2025-08-07,NaN,low,explain your thought process step by step,abstract
3,gpt-5-mini-2025-08-07,NaN,low,explain your thought process step by step,full_text
4,gpt-5-mini-2025-08-07,NaN,high,,abstract
...,...,...,...,...,...
107,deepseek-r1-distill-llama-70b,1.0,NaN,explain your thought process step by step,full_text
108,deepseek-r1-distill-llama-70b,1.5,NaN,,abstract
109,deepseek-r1-distill-llama-70b,1.5,NaN,,full_text
110,deepseek-r1-distill-llama-70b,1.5,NaN,explain your thought process step by step,abstract



Configuration table completed!


# Create LLM reviews

In [ ]:
# @title Run simulation / API calls

# Helper to key a combo row (NaN-safe)
def _nan_to_none(x):

    """Use pandas' isna to catch float('nan'), numpy.nan, pd.NA, and None"""

    try:
        import pandas as pd  # Local import ok in notebooks
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x

def combo_key_tuple(row) -> tuple:

    """Combines the congfigurational setting with None instead of NaN"""

    return (
        row["model"],
        _nan_to_none(row["temperature"]),
        _nan_to_none(row["reasoning_effort"]),
        row["chain_of_thought"],
        row["scope"]
    )

def combo_key_str(row) -> str:

    """Creates readable combo string for logging"""

    t = combo_key_tuple(row)
    # The string format needs to match the tuple elements
    return f"model={t[0]}|temp={t[1]}|re={t[2]}|cot={t[3]}|scope={t[4]}"

# Build the "done" set
done_keys = set()

# Read log-path and update done_keys
if os.path.exists(LOG_PATH):
    with open(LOG_PATH, "r") as f:
        for line in f:
            key = line.strip()
            if key:
                done_keys.add(key)


# Infer completed combos from RESULTS_PATH, but only if fully complete
if os.path.exists(RESULTS_PATH):
    existing_df = pd.read_csv(RESULTS_PATH)
    if not existing_df.empty:
        # Corrected: One row per paper per unique combo, regardless of num_reviews_per_paper
        EXPECT_ROWS_PER_COMBO = len(targets)

        # Normalize NA for comparison
        ex = existing_df.copy()
        # Ensure these columns exist (they should)
        needed = ["model","temperature","reasoning_effort","chain_of_thought","scope"]
        missing_cols = [c for c in needed if c not in ex.columns]
        if not missing_cols:
            # Normalize NAs to None for keys
            for c in ["temperature","reasoning_effort"]:
                ex[c] = ex[c].where(pd.notna(ex[c]), None)

            # Check if all expected review columns are present for each row.
            review_cols_present = all(f'llm_review_{i}' in ex.columns for i in range(1, NUM_REVIEWS_PER_PAPER + 1))

            if review_cols_present:
                # Count rows per combo
                counts = (ex.groupby(needed)
                            .size() # This counts the number of rows for each unique combo
                            .reset_index(name="nrows"))

                # Only mark as done if all expected rows for that combo are present
                for _, r in counts.iterrows():
                    if int(r["nrows"]) == EXPECT_ROWS_PER_COMBO:
                        done_keys.add(combo_key_str(r))

print(f"Already-completed combos: {len(done_keys)}")

# Iterate with resume capability
client = LLMClient(simulate=SIMULATION_ACTIVE)

new_rows_count = 0
for idx, row in grid_df.head(2).iterrows(): # for test reasons only the first two rows
    key = combo_key_str(row)

    if key in done_keys:
        print(f"[SKIP] {key} already complete.")
        continue

    combo = ParamCombo(
        model=row["model"],
        temperature=None if pd.isna(row["temperature"]) else float(row["temperature"]),
        reasoning_effort=None if pd.isna(row["reasoning_effort"]) else str(row["reasoning_effort"]),
        chain_of_thought=row["chain_of_thought"],
        scope=row["scope"]
    )

    print(f"\n[RUN] {idx+1}/{len(grid_df)} -> {key}", flush=True)
    try:
        df_combo = simulate_parameter_combo(combo, targets, client=client)

        # No need to add combo columns here, simulate_parameter_combo already includes them
        # and stores reviews in llm_review_X columns.

        # Save to disc immediately
        if os.path.exists(RESULTS_PATH): # If exists
            df_combo.to_csv(RESULTS_PATH, mode="a", header=False, index=False) # Append
        else: # If not exists
            df_combo.to_csv(RESULTS_PATH, index=False) # Write new
        new_rows_count += len(df_combo) # Increase count

        # Only write after successful completion
        with open(LOG_PATH, "a") as f:
            f.write(key + "\n")

        # Mark as done
        done_keys.add(key)

    except Exception as e:
        print(f"[ERROR] {key} -> {type(e).__name__}: {e}", flush=True)

print("\nProcessing complete!")

In [27]:
# @title Read and process experimental settings

done_keys = set()

if os.path.exists(RESULTS_PATH):
  existing_df = pd.read_csv(RESULTS_PATH)
  if not existing_df.empty:
    # Normalize NA for comparison
    ex = existing_df.copy()
    ex["temperature"] = ex["temperature"].where(pd.notna(ex["temperature"]), None)
    ex["reasoning_effort"] = ex["reasoning_effort"].where(pd.notna(ex["reasoning_effort"]), None)
    done_keys = set(
        ex.groupby(["model","temperature","reasoning_effort","chain_of_thought", "scope"])
        .size()
        .reset_index()[["model","temperature","reasoning_effort","chain_of_thought", "scope"]]
        .apply(lambda r: combo_key_str(r), axis=1)
        .tolist()
        )

print("\nProcessing complete!")


Processing complete!


In [28]:
# @title Check result

# result.csv-file
tvd_mi_results_df = pd.read_csv(RESULTS_PATH)

# Check results
display(tvd_mi_results_df.head(5))

# log.csv-file
tvd_mi_log_df = pd.read_csv(LOG_PATH, header=None, names=['completed_combo'])

# Check logs
display(tvd_mi_log_df.head(5))

print("\nProcessing complete!")

,paper_id,model,temperature,reasoning_effort,chain_of_thought,scope,review_text_0,review_text_1
0,vuD2xEtxZcj,gpt-5-mini-2025-08-07,NaN,low,NaN,abstract,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...","{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem..."
1,WoByU5W5te0,gpt-5-mini-2025-08-07,NaN,low,NaN,abstract,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...","{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem..."
2,XZRmNjUMj0c,gpt-5-mini-2025-08-07,NaN,low,NaN,abstract,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...","{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem..."
3,cDYRS5iZ16f,gpt-5-mini-2025-08-07,NaN,low,NaN,abstract,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...","{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem..."
4,Sh97TNO5YY_,gpt-5-mini-2025-08-07,NaN,low,NaN,abstract,"{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem...","{\n ""model"": ""gpt-5-mini-2025-08-07"",\n ""tem..."


,completed_combo
0,model=gpt-5-mini-2025-08-07|temp=None|re=low|c...
1,model=gpt-5-mini-2025-08-07|temp=None|re=low|c...



Processing complete!


# References
Cummins. J. (2025).
